# ExaHyPE 2 - Getting prepared

This notebook makes sure that your machine is ready for the ExaHyPE 2 tutorials. Work through it once before you start with any other tutorial. When you are done,

- the Peano and ExaHyPE 2 Python API is importable from your notebooks;
- all required third-party Python packages are installed;
- the C++ libraries are built in the flavour that the tutorials link against;
- you have generated, lowered and compiled an (empty) ExaHyPE 2 project;
- you know which generated files you may edit, and how to tame the log output.

![Setup workflow](illustrations/getting-prepared-setup.png)

## Prerequisites

- **System packages and building the C++ libraries.** Follow the @ref page_installation_home "installation instructions". The tutorials need at least the extensions `--enable-blockstructured --enable-exahype` (Autotools) or `-DENABLE_BLOCKSTRUCTURED=ON -DENABLE_EXAHYPE=ON` (CMake). Parallel runs additionally need `--enable-loadbalancing` or `-DENABLE_LOADBALANCING=ON`, respectively.
- **Python environment and Jupyter.** Create a virtual environment, install the requirements and launch Jupyter from within this environment as described on the @ref page_exahype2_tutorials "tutorials overview page".

## Making the Python API visible

Peano's Python API is spread over two source roots: `src` holds `peano4.api`, the new `exahype2.api` and all toolboxes, while `python` still hosts the legacy `exahype2` package that is currently being migrated. Both define a top-level package called `exahype2`, and Python takes whichever it finds first on its search path:

![Python source roots](illustrations/getting-prepared-python-roots.png)

The tutorials use the new API. The cell below therefore puts `src` at the very front of the search path. It does no harm if you have set your `PYTHONPATH` already or installed the API into your virtual environment.

In [ ]:
import os, sys

peano_root = os.path.abspath("../..")
peano_src  = os.path.join(peano_root, "src")
if peano_src in sys.path:
    sys.path.remove(peano_src)
sys.path.insert(0, peano_src)

Every ExaHyPE project starts with the obligatory imports. Peano and ExaHyPE greet you with an ASCII art banner. If you get warnings about invalid escape sequences, you can ignore them: Peano's in-code documentation uses LaTeX formulae, and these confuse some Python versions.

In [ ]:
import peano4.api, exahype2

## Self-check

The following cell checks your setup and tells you what to fix. It also defines two parameters that we use throughout this notebook: the dimension and the build flavour (compile mode). Each compile mode links against libraries with a different postfix, e.g. `libExaHyPE2Core2d_asserts.a` for `ASSERTS`, so the flavour you pick has to be built. Available modes are `RELEASE`, `ASSERTS`, `DEBUG`, `TRACE` and `STATS`.

In [ ]:
import glob, importlib.util

dimensions = 2
mode       = peano4.api.output.CompileMode.ASSERTS

problems = []

print("peano4.api from", os.path.dirname(peano4.api.__file__))
print("exahype2   from", os.path.dirname(exahype2.__file__))
if not hasattr(exahype2, "api"):
    problems.append("exahype2 has been loaded from the legacy python/exahype2 tree: restart the kernel and rerun the path cell first")

for package in ["jinja2", "numpy", "sympy", "mpmath", "matplotlib"]:
    if importlib.util.find_spec(package) is None:
        problems.append("Python package " + package + " is missing: run pip install -r requirements.txt within your virtual environment")

if os.path.isfile(os.path.join(peano_root, "config.log")):
    print("build      : Autotools (found config.log)")
    build_directories = [os.path.join(peano_root, "src")]
elif "PEANO_CMAKE_BUILD_DIR" in os.environ:
    print("build      : CMake (environment variable PEANO_CMAKE_BUILD_DIR)")
    build_directories = [os.environ["PEANO_CMAKE_BUILD_DIR"]]
else:
    build_directories = [
        directory for directory in glob.glob(os.path.join(peano_root, "*"))
        if os.path.isfile(os.path.join(directory, "PeanoTargets.cmake"))
    ]
    print("build      : CMake" if build_directories else "build      : none found")
if not build_directories:
    problems.append("no Peano build found: run configure and make (Autotools) or cmake and make within a build subdirectory of " + peano_root)

postfix = {"RELEASE": "", "ASSERTS": "_asserts", "DEBUG": "_debug", "TRACE": "_trace", "STATS": "_stats"}[mode.name]
library = "libExaHyPE2Core" + str(dimensions) + "d" + postfix + ".a"
found   = [match for directory in build_directories for match in glob.glob(os.path.join(directory, "**", library), recursive=True)]
if build_directories and not found:
    problems.append(library + " not found: enable ExaHyPE 2 in your build, run make, or pick another mode")
for match in found:
    print("library    :", match)

print()
print("All set." if not problems else "Please fix:\n- " + "\n- ".join(problems))

## Creating an ExaHyPE and Peano project

Every ExaHyPE project defines a sequence of lowering steps. That is, we start with an ExaHyPE project object. This represents the domain level, where we do the maths. After that, we lower this project into a Peano project. This is a technical representation (still in Python) of the arising algorithm, i.e. something that represents the actual code base realising our domain knowledge. Finally, we ask Peano to lower this project into native C++ code plus a Makefile.

![Lowering process](illustrations/generic-python-workflow.png)

The term lowering stems from the compiler community, where it is used to discuss different domain-specific languages and their abstraction levels.

The generator produces quite a lot of files and directories. We therefore ask it to write everything into a subdirectory `00_getting-started`.

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2"],
    project_name = "gettingprepared",
    executable   = "my-code",
    directory    = "00_getting-started",
)

## Configure the global simulation

Before we lower the project, we tell it which domain to work on and how long to simulate. These parameters end up as constants in the generated C++ code, so they have to be set **before** the lowering. We use the unit square here.

In [ ]:
project.set_global_simulation_parameters(
    dimensions            = dimensions,
    offset                = [0.0 for _ in range(dimensions)],
    size                  = [1.0 for _ in range(dimensions)],
    min_end_time          = 1.0,
    max_end_time          = 1.0,
    periodic_BC           = [True for _ in range(dimensions)],
)

Real projects also add solvers at this point. We skip this here, as we only want to check the workflow.

## Lowering into C++

`set_Peano4_installation()` takes Peano's root directory plus the compile mode. It inspects your build and picks up compiler and linker settings from it:

- **Autotools:** it parses `config.log` and `src/Makefile` in the root directory;
- **CMake:** it searches the root directory for a subdirectory holding `PeanoTargets.cmake`, such as `build`. If your build directory lives elsewhere, point the environment variable `PEANO_CMAKE_BUILD_DIR` to it.

`generate_Peano4_project()` then lowers the ExaHyPE project into a Peano project, and `generate()` dumps the C++ code plus a Makefile.

In [ ]:
project.set_Peano4_installation(peano_root, mode)

peano4_project = project.generate_Peano4_project()
peano4_project.generate()

## What has been generated

The generated code follows the generic @ref page_architecture_home of any code built on top of Peano:

![System architecture](illustrations/architecture-static.png)

A domain-specific code sits on top of some extensions plus the Peano core, and all of the components use a common technical architecture. Let's look at what we got:

In [ ]:
!ls 00_getting-started

- `Makefile` builds the executable. Use it on the command line or from within the notebook.
- `Constants.h` holds the global simulation parameters from above.
- `gettingprepared-main.cpp` and `gettingprepared-main.h` are the main routine.
- `repositories` and `runners` hold the glue code that drives the mesh traversals.
- `.gitignore` lists all generated files, so you can put your project under version control right away.
- `README-*.md` summarises how the project has been configured, and `log.md` logs what the Python API has done. Both are named after the script that ran the lowering. Within a notebook, this is Jupyter's kernel launcher.

All of these files are regenerated whenever you rerun the lowering, so do not edit them by hand. Once you add a solver, ExaHyPE also generates the solver's implementation files. These are created only once and never overwritten: they are yours to edit. `make clean` removes the build output and the generated sources, but leaves your solver implementation alone.

Now compile the code:

In [ ]:
!cd 00_getting-started && make -j4

There is nothing to run yet: our project has no solver, so there is nothing to step through. We run our first simulation in the next tutorial.

## Taming the log output

ExaHyPE codes can be very chatty. On startup, the executable reads a log filter file called `exahype2.log-filter` from its working directory. If the file is missing, the code warns and prints everything. Use `set_log_filter_file()` on the project if you prefer a different file name. Each line of the file reads

    level   namespace   rank   black|white   phase

where `rank` -1 means all ranks, and `white` switches the matching messages on and `black` switches them off. The cell below writes a filter that shows ExaHyPE's messages, but hides those of the Peano core, the technical architecture and the toolboxes. See @ref tarch_logging for all details. There are many further examples in the repository: search for files ending in `log-filter`.

In [ ]:
%%writefile 00_getting-started/exahype2.log-filter
all   tarch      -1   black   always-on
all   peano4     -1   black   always-on
all   exahype2   -1   white   always-on
all   toolbox    -1   black   always-on

## Troubleshooting

- **No module named peano4:** the path cell has not been run, or the notebook does not sit in `tutorials/exahype2`. Adapt `peano_root` in the path cell.
- **No module named jinja2 (or numpy, ...):** Jupyter does not run within your virtual environment. Launch Jupyter from the activated environment or pick the environment's kernel.
- **module exahype2 has no attribute api, or an ImportError deep inside python/exahype2:** the legacy package has been picked up. Restart the kernel and rerun the path cell before any other cell.
- **please hand over directory where ./configure had been called:** `peano_root` points to the wrong directory, or Peano has not been configured or built yet.
- **Linker error cannot find -lExaHyPE2Core2d_asserts (or similar):** the library flavour for the chosen mode has not been built. Build it or change `mode` and rerun the lowering.

## What's next

- Continue with @ref tutorials_exahype2_10_euler, which adds your first solver, then builds and runs it.
- Change `dimensions` to 3 in the self-check cell and rerun the notebook: the self-check tells you whether you have the 3d libraries, too.
- Look up the signature of `set_global_simulation_parameters()` in the documentation of `exahype2.api.Project` to see what else you can configure.